# Arithmetic TNFR: Interactive Visualizations

This notebook is a companion to the guide: [ARITHMETIC_TNFR.md](../ARITHMETIC_TNFR.md).

It visualizes ΔNFR, local coherence, and structural fields (φ, |∇φ|, K_φ, Φ_s, ξ_C) on the arithmetic network, plus operators demo (UM coupling + RA resonance propagation).

In [ ]:
# Setup: imports and path configuration
import os, sys, math
import numpy as np
import matplotlib.pyplot as plt

# Ensure local 'src' is importable when running from docs/notebooks
repo_src = os.path.abspath(os.path.join('..', '..', 'src'))
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

from tnfr.mathematics.number_theory import ArithmeticTNFRNetwork

plt.style.use('seaborn-v0_8')
%matplotlib inline

In [ ]:
# Parameters
MAX_NUMBER = 200
DELTA_NFR_THRESHOLD = 0.20
PHASE_METHOD = 'logn'  # 'logn' | 'spectral' | 'nuf'

In [ ]:
# Build the arithmetic TNFR network and compute structural fields
net = ArithmeticTNFRNetwork(max_number=MAX_NUMBER)
validation = net.validate_prime_detection(delta_nfr_threshold=DELTA_NFR_THRESHOLD)
fields = net.compute_structural_fields(phase_method=PHASE_METHOD)
phi = fields['phi']
phi_grad = fields['phi_grad']
k_phi = fields['k_phi']
phi_s = fields['phi_s']
xi_c = fields['xi_c']
validation

## ΔNFR by n (prime candidates near 0)

In [ ]:
# Scatter plot of ΔNFR vs n
ns = sorted(net.graph.nodes())
deltas = [net.graph.nodes[n]['DELTA_NFR'] for n in ns]
is_prime = [net.graph.nodes[n].get('is_prime', False) for n in ns]

plt.figure(figsize=(10, 4))
plt.axhline(0.0, color='gray', lw=1, ls='--')
plt.scatter(ns, deltas, c=['tab:green' if p else 'tab:red' for p in is_prime], s=18, alpha=0.8)
plt.title('ΔNFR by n (green = prime, red = composite)')
plt.xlabel('n')
plt.ylabel('ΔNFR(n)')
plt.show()

## Local coherence histogram

In [ ]:
# Histogram of local coherence C(n) = 1 / (1 + |ΔNFR(n)|)
coh = [net.graph.nodes[n]['coherence_local'] for n in ns]
plt.figure(figsize=(8, 4))
plt.hist(coh, bins=20, color='tab:blue', alpha=0.8)
plt.title('Local Coherence Distribution')
plt.xlabel('C(n)')
plt.ylabel('Frequency')
plt.show()

## Phase and gradient/curvature

In [ ]:
# Line plots for φ(n), |∇φ|(n), and |K_φ|(n)
phis = [phi.get(n, np.nan) for n in ns]
grads = [phi_grad.get(n, np.nan) for n in ns]
curvs = [abs(k_phi.get(n, np.nan)) for n in ns]

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(ns, phis, color='tab:purple')
axes[0].set_ylabel('φ(n)')
axes[0].set_title(f'Phase φ (method={PHASE_METHOD})')

axes[1].plot(ns, grads, color='tab:orange')
axes[1].set_ylabel('|∇φ|(n)')
axes[1].set_title('Phase Gradient |∇φ|')

axes[2].plot(ns, curvs, color='tab:green')
axes[2].set_ylabel('|K_φ|(n)')
axes[2].set_title('Phase Curvature Magnitude |K_φ|')
axes[2].set_xlabel('n')

plt.tight_layout()
plt.show()

## Structural potential Φ_s and coherence length ξ_C

In [ ]:
# Φ_s as a line plot (global structural potential per node) and ξ_C summary
phi_s_vals = [phi_s.get(n, np.nan) for n in ns]
plt.figure(figsize=(10, 4))
plt.plot(ns, phi_s_vals, color='tab:brown')
plt.title('Structural Potential Φ_s (α=2)')
plt.xlabel('n')
plt.ylabel('Φ_s(n)')
plt.show()

print('Coherence Length ξ_C (summary):')
for k, v in xi_c.items():
    print(f'  {k}: {v}')

## TNFR Operators Demo

This section demonstrates the two structural operators on the arithmetic network:

- **UM (Coupling)**: Creates structural links via phase synchronization. Requires phase verification: |φᵢ − φⱼ| ≤ Δφ_max (U3 grammar).
- **RA (Resonance)**: Amplifies and propagates patterns coherently. Tracks propagation history and metrics.

In [ ]:
# UM Operator: Coupling with phase verification (U3)
# Ensure phases are computed and stored
net.compute_phase(method='logn', store=True)

# Apply coupling with delta_phi_max = π/2
delta_phi_max = math.pi / 2
coupled_edges = net.apply_coupling(delta_phi_max=delta_phi_max)

print(f'UM Operator (Coupling):')
print(f'  delta_phi_max: {delta_phi_max:.3f} rad ({delta_phi_max*180/math.pi:.1f}°)')
print(f'  Coupled edges created: {len(coupled_edges)}')
print(f'  Grammar U3 compliance: phase verification required for all couplings')
print(f'  Sample coupled pairs (first 10):')
for i, (n1, n2) in enumerate(coupled_edges[:10], 1):
    phi1 = net.graph.nodes[n1].get('phi', 0.0)
    phi2 = net.graph.nodes[n2].get('phi', 0.0)
    delta_phi = abs(phi1 - phi2)
    print(f'    {i}. ({n1:3d}, {n2:3d}): Δφ = {delta_phi:.3f} rad (✓ ≤ {delta_phi_max:.3f})')

In [ ]:
# RA Operator: Resonance propagation from primes
# Start resonance from all primes, propagate for 3 steps
steps = 3
gain = 1.0  # resonance amplification
decay = 0.1  # per-step decay

history = net.resonance_from_primes(steps=steps, gain=gain, decay=decay)

print(f'\nRA Operator (Resonance):')
print(f'  Steps: {steps}')
print(f'  Gain: {gain}, Decay: {decay}')
print(f'  Source: all primes in network')
print(f'  Propagation history length: {len(history)}')

# Get metrics from final state
if history:
    metrics = net.resonance_metrics(history[-1])
    print(f'\n  Resonance Metrics (final state):')
    print(f'    Total nodes activated: {metrics.get("total_activated", 0)}')
    print(f'    Mean resonance amplitude: {metrics.get("mean_amplitude", 0.0):.4f}')
    print(f'    Max resonance amplitude: {metrics.get("max_amplitude", 0.0):.4f}')
    print(f'    Effective coupling strength: {metrics.get("mean_coupling_strength", 0.0):.4f}')
    print(f'    Phase coherence: {metrics.get("phase_coherence", 0.0):.4f}')

In [ ]:
# Visualize resonance propagation over time
if len(history) > 1:
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    
    # Top: resonance amplitude by node at each step
    for step_idx, resonance_state in enumerate(history):
        amplitudes = [resonance_state.get(n, 0.0) for n in ns]
        axes[0].plot(ns, amplitudes, alpha=0.5, label=f'Step {step_idx}')
    axes[0].set_ylabel('Resonance amplitude')
    axes[0].set_title('Resonance Propagation from Primes')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Bottom: cumulative activation count
    activation_counts = []
    for resonance_state in history:
        count = sum(1 for n in ns if resonance_state.get(n, 0.0) > 0.01)
        activation_counts.append(count)
    axes[1].plot(range(len(activation_counts)), activation_counts, marker='o', color='tab:red')
    axes[1].set_xlabel('Propagation step')
    axes[1].set_ylabel('Active nodes (amp > 0.01)')
    axes[1].set_title('Network Activation Over Steps')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('Resonance history too short for visualization')

## Operators Summary

**UM (Coupling)**:
- Creates structural links between nodes with compatible phases
- Grammar U3 enforcement: only couples if |φᵢ − φⱼ| ≤ Δφ_max
- Enables information exchange and network synchronization

**RA (Resonance)**:
- Amplifies and propagates coherent patterns
- Tracks multi-step propagation history
- Metrics: activation, amplitude, coupling strength, phase coherence
- Preserves EPI identity (does not alter structural form)

Both operators are **read-only** for phase fields (U6 compliance) and follow the unified grammar (U1-U5).